# SHAP Explainability — Fraud Detection Model

## What is SHAP?

SHAP (SHapley Additive exPlanations) answers one question: **for a specific prediction, how much did each feature contribute to the score?**

The model output a fraud probability. SHAP decomposes that probability into per-feature contributions. For example:
- `newbalanceOrig = 0` pushed the score **up** by 0.35 (strong fraud signal — account drained)
- `amount = 181` pushed the score **down** by 0.02 (small amount, less suspicious)

This is useful in two ways:
1. **Global** — run SHAP on many transactions to understand what the model has generally learned. Validates that the model is using sensible features, not spurious correlations.
2. **Local** — run SHAP on one transaction to explain why it was flagged. Required in production for compliance, dispute resolution, and analyst review.

---

## How many data points do you run SHAP on?

This depends on what you are doing:

| Use case | How many rows |
|---|---|
| Global summary plots (feature importance, beeswarm) | Sample of 1,000–5,000 rows |
| Local explanation (one transaction) | 1 row |
| Production — explain every scored transaction | All rows, one at a time at inference |

We are using `TreeExplainer`, which is the SHAP explainer built specifically for tree-based models (XGBoost, LightGBM, Random Forest). It is **exact**, not approximate — it does not estimate SHAP values, it computes them precisely using the tree structure. This makes it fast enough to use in production.

For global plots we will sample 2,000 rows from the holdout set — enough to get a representative picture without waiting a long time. The sample will be stratified (equal proportion of fraud and non-fraud) so the plots are not dominated by the majority class.

## Step 1 — Imports

In [ ]:
import shap
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# shap.initjs() enables interactive JavaScript plots in the notebook.
# Only needed for force plots (the single-prediction visualisation).
shap.initjs()

## Step 2 — Load the model

The model was saved as a dictionary `{'model': XGBClassifier, 'study': ..., ...}` by the training notebook.
We extract just the classifier.

In [ ]:
_loaded = joblib.load("../model/XGBOOST_FULL.joblib")

# The training script saved a dict — the actual XGBClassifier is under the 'model' key
model = _loaded['model']

print("Model type:", type(model))
print("Number of trees:", model.n_estimators)

## Step 3 — Recreate the holdout set

We reproduce the exact same train/holdout split used in training (same `random_state=42`, same `test_size=0.3`).
This guarantees we are running SHAP on data the model has **never seen**. 

If you ran SHAP on training data, the explanations would be misleading — the model has memorised those rows to some extent.

In [ ]:
data = pd.read_parquet("/Users/prathikkundaragi/MY PROJECTS/Fraud Dataset/model_data_full.parquet")

# Reproducing the exact same split as in 02_xgboost_training.ipynb
# random_state=42 and test_size=0.3 must match exactly, otherwise the holdout is different
_, data_holdout = train_test_split(data, test_size=0.3, random_state=42)

X_holdout = data_holdout.drop('isFraud', axis=1)
y_holdout = data_holdout['isFraud']

print(f"Holdout size : {len(X_holdout):,} rows")
print(f"Fraud cases  : {y_holdout.sum():,}  ({y_holdout.mean()*100:.2f}%)")
print(f"Features     : {list(X_holdout.columns)}")

## Step 4 — Stratified sample for global SHAP

The holdout has ~2.5 million rows. Running SHAP on all of them is unnecessary — the global plots stabilise after a few thousand samples.

We take 2,000 rows but keep the same fraud rate as the full holdout. This is called **stratified sampling**.

Why stratify? If we sampled randomly, we might get 0 or very few fraud cases (fraud is rare — ~0.14% of transactions). With no fraud cases in the sample, the beeswarm plot would only show what drives the score down, not up.

In [ ]:
# We want 2000 rows total, but with enough fraud cases to be meaningful.
# Instead of keeping the natural ~0.14% fraud rate (which gives ~3 fraud rows in 2000),
# we oversample fraud to 10% of the sample. This is only for visualisation — it does
# not affect the model or the SHAP values themselves.

N_SAMPLE   = 2000
FRAUD_FRAC = 0.10   # 10% fraud in the sample = 200 fraud rows

n_fraud  = int(N_SAMPLE * FRAUD_FRAC)          # 200 fraud rows
n_legit  = N_SAMPLE - n_fraud                   # 1800 legit rows

fraud_idx = y_holdout[y_holdout == 1].index
legit_idx = y_holdout[y_holdout == 0].index

# sample() draws rows at random without replacement; random_state fixes the seed for reproducibility
fraud_sample = X_holdout.loc[fraud_idx].sample(n=n_fraud, random_state=42)
legit_sample = X_holdout.loc[legit_idx].sample(n=n_legit, random_state=42)

# Combine and shuffle so fraud and legit rows are interleaved
X_sample = pd.concat([fraud_sample, legit_sample]).sample(frac=1, random_state=42)
y_sample = y_holdout.loc[X_sample.index]

print(f"Sample size  : {len(X_sample)}")
print(f"Fraud in sample : {y_sample.sum()} rows ({y_sample.mean()*100:.1f}%)")

## Step 5 — Compute SHAP values

`TreeExplainer` works by traversing every tree in the XGBoost ensemble and computing the exact contribution of each feature to the final prediction. It is the only SHAP explainer that gives exact (not approximate) values.

`explainer(X_sample)` returns a SHAP `Explanation` object. Think of it as a matrix of the same shape as `X_sample` — one SHAP value per row per feature. A **positive** SHAP value means that feature pushed the fraud score **up**. A **negative** value means it pushed it **down**.

In [ ]:
# TreeExplainer is initialised with the trained model.
# It reads the tree structure once here — this is the slow step (a few seconds).
explainer = shap.TreeExplainer(model)

# Now compute SHAP values for all 2000 rows in our sample.
# shap_values is an Explanation object with shape (2000, 7) — 2000 rows, 7 features.
# This takes a few seconds depending on the number of trees (450 in our model).
shap_values = explainer(X_sample)

print("SHAP values shape:", shap_values.values.shape)
print("Features          :", list(X_sample.columns))
print()
print("Example — first row SHAP values:")
for feat, val in zip(X_sample.columns, shap_values.values[0]):
    direction = "FRAUD" if val > 0 else "legit"
    print(f"  {feat:<20} {val:+.4f}  → pushes score {direction}")

## Step 6 — Global: Beeswarm plot (the most important SHAP chart)

This is the standard chart everyone means when they say "SHAP plot".

**How to read it:**
- Each row is one feature. Features are sorted top to bottom by importance (most important at top).
- Each dot is one transaction from the sample.
- **X-axis (horizontal position):** the SHAP value — how much that feature moved the fraud score. Far right = pushed toward fraud. Far left = pushed away from fraud.
- **Colour:** the actual feature value. Red = high value. Blue = low value.

**What you are looking for:** Do the patterns make business sense? For example, if `newbalanceOrig` is red (high balance remaining) and sitting on the left (pushing score down), that means high balance after the transaction reduces fraud probability — which makes sense, fraud usually drains the account.

In [ ]:
plt.figure()

# summary_plot with plot_type="dot" is the beeswarm.
# shap_values contains the SHAP matrix; X_sample provides the actual feature values for colouring.
# max_display=7 shows all 7 features (our model has exactly 7).
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="dot",
    max_display=7,
    show=False
)

plt.title("SHAP Beeswarm — Global Feature Impact on Fraud Score", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("../model/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to model/shap_beeswarm.png")

## Step 7 — Global: Mean absolute SHAP (bar chart)

The beeswarm shows direction and distribution but can be visually busy. The bar chart collapses it to a single number per feature: the **mean absolute SHAP value** across all rows.

Think of it as: on average, how much does this feature move the fraud score, regardless of direction? This is the cleanest way to rank feature importance.

This is what you would put in a report to a non-technical stakeholder: "these are the top 3 features driving the model".

In [ ]:
plt.figure()

shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="bar",   # bar chart instead of beeswarm
    max_display=7,
    show=False
)

plt.title("Mean Absolute SHAP Value — Feature Importance", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("../model/shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

# Also print as a table — useful for reports
mean_shap = pd.DataFrame({
    'feature'         : X_sample.columns,
    'mean_abs_shap'   : np.abs(shap_values.values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print("\nFeature importance (SHAP):")
print(mean_shap.to_string(index=False))

## Step 8 — Local: Explain a single high-score fraud transaction

This is the "why was this transaction flagged?" question. This is what you would show to:
- A compliance officer reviewing a blocked transaction
- A customer service agent explaining to a customer why their payment was declined
- A risk analyst validating whether the flag makes sense before taking action

We pick the transaction in the holdout with the **highest fraud score** — the one the model is most confident about.

**Waterfall plot explained:**
- The bar starts from `E[f(x)]` — the model's average output across all transactions (the baseline)
- Each feature either adds (red, pushes right) or subtracts (blue, pushes left) from that baseline
- The final value at the top is the fraud probability for this specific transaction
- Features are ordered by how much they contributed to this individual prediction

In [ ]:
# Find the transaction with the highest fraud score in the full holdout
fraud_scores_holdout = model.predict_proba(X_holdout)[:, 1]
top_fraud_idx = np.argmax(fraud_scores_holdout)    # position (integer index) of the max score

print("=== Highest-scored transaction ===")
print(f"Fraud score : {fraud_scores_holdout[top_fraud_idx]:.6f}")
print(f"Actual label: {y_holdout.iloc[top_fraud_idx]} (1=fraud, 0=legit)")
print()
print("Feature values:")
print(X_holdout.iloc[top_fraud_idx])

In [ ]:
# Compute SHAP values for just this one row.
# We pass a single-row DataFrame (not a Series) because the explainer expects 2D input.
single_row = X_holdout.iloc[[top_fraud_idx]]
shap_single_fraud = explainer(single_row)

plt.figure()
# waterfall plot: index 0 because we passed one row (index 0 of the result)
shap.plots.waterfall(shap_single_fraud[0], show=False)
plt.title("Waterfall — Why was this transaction flagged as fraud?", fontsize=12, pad=15)
plt.tight_layout()
plt.savefig("../model/shap_waterfall_fraud.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9 — Local: Explain a low-score legitimate transaction

Same plot, opposite end. We pick the transaction the model is **most confident is not fraud**.

This is useful for calibration — you want to see that the reasons it gives for a legitimate score also make business sense. If the model says "low score because `amount` is small", that might be right on average but could be wrong for specific fraud patterns (structuring — many small transactions). SHAP local explanations help you catch these blind spots.

In [ ]:
# Transaction with the lowest fraud score (most confidently legitimate)
bottom_legit_idx = np.argmin(fraud_scores_holdout)

print("=== Lowest-scored transaction ===")
print(f"Fraud score : {fraud_scores_holdout[bottom_legit_idx]:.6f}")
print(f"Actual label: {y_holdout.iloc[bottom_legit_idx]} (1=fraud, 0=legit)")
print()
print("Feature values:")
print(X_holdout.iloc[bottom_legit_idx])

single_row_legit = X_holdout.iloc[[bottom_legit_idx]]
shap_single_legit = explainer(single_row_legit)

plt.figure()
shap.plots.waterfall(shap_single_legit[0], show=False)
plt.title("Waterfall — Why was this transaction scored as legitimate?", fontsize=12, pad=15)
plt.tight_layout()
plt.savefig("../model/shap_waterfall_legit.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 10 — Dependence plot: how one feature interacts with the score

A dependence plot picks one feature and plots its actual value (x-axis) against its SHAP contribution (y-axis) across all rows in the sample. It shows the relationship the model has learned — is it linear, step-function, or something else?

The colour of each dot shows a second feature chosen automatically by SHAP because it has the strongest interaction with the main feature.

We plot `newbalanceOrig` — the account balance after the transaction. Expectation: when the balance goes to zero, that is a strong fraud signal (account drained). The plot should show high positive SHAP values (pushing toward fraud) when `newbalanceOrig` is near 0.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: newbalanceOrig vs SHAP contribution
# "interaction_index='auto'" lets SHAP pick the best colouring feature automatically
shap.dependence_plot(
    "newbalanceOrig",
    shap_values.values,
    X_sample,
    interaction_index="auto",
    ax=axes[0],
    show=False
)
axes[0].set_title("newbalanceOrig — SHAP dependence", fontsize=11)

# Right plot: oldbalanceOrg (balance before the transaction)
shap.dependence_plot(
    "oldbalanceOrg",
    shap_values.values,
    X_sample,
    interaction_index="auto",
    ax=axes[1],
    show=False
)
axes[1].set_title("oldbalanceOrg — SHAP dependence", fontsize=11)

plt.tight_layout()
plt.savefig("../model/shap_dependence.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 11 — How SHAP connects to production

The SHAP explainer is loaded once at API startup (same as the model) and runs on every incoming transaction. The `/predict` endpoint returns the fraud score alongside per-feature SHAP contributions — so the caller always knows not just *how suspicious* the transaction is, but *why*.

See `app/model.py` for the implementation.